# PBO — Polaridad selectiva

**Trabajo:** `MVC / selective-polarity / seed-0`.

Usa las instancias MVC en bruto y sus etiquetas `.backbone` ya preparadas para generar la transformación selectiva solo en train; conserva los artefactos en `PBO-Tesis/runs/MVC/selective-polarity/seed-0`. Entrena BackPaS GTR de 8 capas durante 65 épocas, con batch 64. La transformación deriva las etiquetas desde las existentes y no ejecuta Gurobi para reetiquetar. No volver a ejecutar si el trabajo ya está activo, completado o fallido; primero revisar su estado y corregir esta misma celda si corresponde.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import json, os, shutil, subprocess, sys

DRIVE_ROOT = Path('/content/drive/MyDrive/PBO-Tesis')
REPO = Path('/content/backbone-pbo')
if not (REPO / 'pyproject.toml').is_file():
    subprocess.run(['git', 'clone', 'https://github.com/jgomezaTI/PBO-DA.git', str(REPO)], check=True)

UPSTREAMS = [
    ('backpas', 'https://github.com/bryan-alvarado-ulloa/backpas.git',
     'b5c633579fa73d673f3c4e65e177c89bcddea3d7'),
    ('guroback', 'https://github.com/bryan-alvarado-ulloa/guroback.git',
     '428f7d112aa82a3b6b951793e1b9e6b712ef1cea'),
]
for name, url, commit in UPSTREAMS:
    dest = REPO / 'external' / name
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', url, str(dest)], check=True)
        subprocess.run(['git', '-C', str(dest), 'checkout', '--detach', commit], check=True)

DATA = REPO / 'data/backpas'
DATA.mkdir(parents=True, exist_ok=True)
DRIVE_DATA = DRIVE_ROOT / 'data/backpas'
PROBLEM = 'MVC'  # Fijado para este notebook: no cambiar a MIS/CA por accidente.
raw_source = DRIVE_DATA / PROBLEM
if not (raw_source / 'instance').is_dir() or not (raw_source / 'backbone').is_dir():
    raise FileNotFoundError(
        f'Faltan carpetas de datos en bruto instance/ y backbone/: {raw_source}')
dest = DATA / PROBLEM
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(raw_source, dest)
for required_file in ('partitions.json', 'source.json'):
    if not (dest / required_file).is_file():
        raise FileNotFoundError(f'Falta {required_file} en {dest}')
instance_count = sum(1 for path in (dest / 'instance').iterdir() if path.is_file())
backbone_count = sum(1 for path in (dest / 'backbone').iterdir() if path.is_file())
if (instance_count, backbone_count) != (900, 900) and PROBLEM == 'MVC':
    raise RuntimeError(f'MVC en bruto incompleto: {instance_count} instancias, '
                       f'{backbone_count} etiquetas')
print(f'{PROBLEM} en bruto: {instance_count} instancias, {backbone_count} etiquetas')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml,backpas,data]'],
               cwd=REPO, check=True)

# The authoritative experiment matrix is generated from the repository code.
subprocess.run([sys.executable, 'scripts/materialize_pending_augmentations.py',
                '--repo', str(REPO)], cwd=REPO, check=True)

TECHNIQUE = 'selective-polarity'
AUGMENTATION_KIND = 'selective-polarity'

MATRIX = REPO / 'configs/pending_augmentation_jobs.json'
matrix = json.loads(MATRIX.read_text(encoding='utf-8'))
def job_state(job):
    problem, technique, seed = job.split('/')
    work = DRIVE_ROOT / 'runs' / problem / technique / seed
    if any((work / marker).is_file() for marker in ('metrics.json', 'experiment-status.json')):
        return 'completed'
    status_path = work / 'status.json'
    if status_path.is_file():
        try:
            state = json.loads(status_path.read_text(encoding='utf-8')).get('state')
        except (OSError, json.JSONDecodeError):
            return 'unknown'
        if state == 'completed':
            return 'completed'
        if state in {'generating', 'preparing', 'training', 'running'}:
            return state
        if state == 'failed':
            return 'failed'
        return 'unknown'
    return 'available'

ALL_JOBS = {row['problem']: f"{row['problem']}/{row['technique']}/seed-{row['seed']}"
            for row in matrix['jobs'] if row['technique'] == TECHNIQUE}
STATES = {problem: job_state(job) for problem, job in ALL_JOBS.items()}
TECHNIQUE_ORDER = ['selective-polarity', 'constraint-perturbation',
                   'objective-perturbation', 'synthetic-conditioned']
TECHNIQUE_INDEX = TECHNIQUE_ORDER.index(TECHNIQUE)
PROBLEM_ORDER = ['MVC', 'MIS', 'CA']
PREFERRED = PROBLEM_ORDER[TECHNIQUE_INDEX % len(PROBLEM_ORDER)]
print('Técnica:', TECHNIQUE)
print('Estado actual de la matriz para esta técnica:')
for problem, state in STATES.items():
    print(f'  {problem}/{TECHNIQUE}/seed-0: {state}')
AVAILABLE = [problem for problem, state in STATES.items() if state == 'available']
if not AVAILABLE:
    raise RuntimeError('No hay trabajos disponibles para esta técnica; los existentes están activos o completados.')
print('Problemas disponibles:', ', '.join(AVAILABLE))
if PREFERRED in AVAILABLE:
    print(f'Asignación sugerida para mantener variedad entre técnicas: {PREFERRED}')
if PROBLEM not in AVAILABLE:
    raise ValueError(f"{PROBLEM!r} no está disponible. Elige uno de: {', '.join(AVAILABLE)}")
JOBS = [ALL_JOBS[PROBLEM]]
print('Trabajo elegido:', JOBS[0])
print('Checkpoint:', DRIVE_ROOT / 'runs' / JOBS[0])
if TECHNIQUE == 'objective-perturbation':
    cached = DRIVE_ROOT / 'runs/generation-cache/CA-objective-perturbation.partial'
    local = REPO / 'data/backpas/.CA-objective-perturbation-v1.partial'
    if cached.is_dir() and not local.exists():
        shutil.copytree(cached, local)
        print('Restaurada generación parcial de CA desde Drive.')


In [ ]:
# Ejecuta exactamente MVC / selective-polarity / seed-0.
# Esta celda materializa y verifica la transformacion antes de iniciar BackPaS.
import csv, json, subprocess, sys, torch

JOB = 'MVC/selective-polarity/seed-0'
MATRIX = REPO / 'configs/pending_augmentation_jobs.json'
config = json.loads(MATRIX.read_text(encoding='utf-8'))
job = next(j for j in config['jobs'] if j['problem'] == 'MVC'
           and j['technique'] == 'selective-polarity' and j['seed'] == 0)
DATASET = REPO / job['dataset']
SOURCE = REPO / job['source_dataset']
WORK = DRIVE_ROOT / 'runs/MVC/selective-polarity/seed-0'

# No repetir una corrida completa, activa, fallida o con estado ambiguo.
markers = ('metrics.json', 'experiment-status.json')
if any((WORK / marker).is_file() for marker in markers):
    raise RuntimeError(f'No se ejecuta: ya hay resultado en {WORK}. Revisalo primero.')
status_path = WORK / 'status.json'
if status_path.is_file():
    status = json.loads(status_path.read_text(encoding='utf-8'))
    state = status.get('state')
    if state in {'completed', 'generating', 'preparing', 'training', 'running', 'failed'}:
        raise RuntimeError(f'No se ejecuta: status.json indica state={state!r}.')
    if state not in (None, 'available'):
        raise RuntimeError(f'No se ejecuta: estado desconocido en status.json: {state!r}.')
checkpoint_candidates = list(WORK.rglob('best_model.pth')) + list(WORK.rglob('last_model.pth'))
if checkpoint_candidates:
    raise RuntimeError(f'No se ejecuta: ya hay checkpoints en {WORK}; revisar corrida antes.')

# Aplicar selective polarity solo si aun no existe un dataset transformado.
manifest_path = DATASET / 'augmentation.json'
if not manifest_path.is_file():
    if DATASET.exists():
        raise RuntimeError(f'Dataset parcial o sin manifiesto: {DATASET}; no se borra ni sobrescribe.')
    generator = [sys.executable, 'scripts/materialize_perturbation.py',
                 str(SOURCE), str(DATASET), '--kind', 'selective-polarity',
                 '--seed', str(job['seed']), '--replicas', str(job.get('replicas', 1)),
                 '--attempts', str(job.get('attempts', 20))]
    print('Aplicando transformacion selective-polarity sobre MVC/train:', flush=True)
    print(' '.join(generator), flush=True)
    subprocess.run(generator, cwd=REPO, check=True)

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('kind') != 'selective-polarity' or manifest.get('generated', 0) < 1:
    raise RuntimeError('Se cancela: el dataset no es una transformacion selective-polarity valida.')
if manifest.get('seed') != job['seed']:
    raise RuntimeError(f"Seed del manifiesto no coincide: {manifest.get('seed')} != {job['seed']}")
if manifest.get('replicas_per_train_instance') != job.get('replicas', 1):
    raise RuntimeError('La cantidad de replicas del manifiesto no coincide con la matriz.')
if not (DATASET / 'instance').is_dir() or not (DATASET / 'backbone').is_dir():
    raise RuntimeError('Dataset transformado incompleto: faltan instance/ o backbone/.')
print(f"Transformacion verificada: {manifest['generated']} instancias; "
      f"etiquetado={manifest.get('label_solver')}; transformación algebraica, sin solver adicional.")

# Mantener el protocolo de entrenamiento indicado para BackPaS GTR.
config['defaults'].update({'epochs': 65, 'threads': 4, 'batch_size': 64,
                           'batch_accumulation_size': 64, 'cuda': True})
MATRIX.write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
runner = REPO / 'scripts/run_backpas_seeded.py'
runner_source = runner.read_text(encoding='utf-8')
runner_source = runner_source.replace(
    'torch.use_deterministic_algorithms(True)',
    'torch.use_deterministic_algorithms(True, warn_only=True)')
if 'CUBLAS_WORKSPACE_CONFIG' not in runner_source:
    runner_source = runner_source.replace(
        'os.environ["MKL_NUM_THREADS"] = str(args.threads)',
        'os.environ["MKL_NUM_THREADS"] = str(args.threads)\n'
        '    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")')
runner.write_text(runner_source, encoding='utf-8')
print('Entrenador: BackPaS GTR, 8 capas; epocas=65, batch=64, acumulacion=64, seed=0.')
command = [sys.executable, 'scripts/cloud_run.py', '--repo', str(REPO),
           '--matrix', 'configs/pending_augmentation_jobs.json', '--job', JOB,
           '--drive-root', str(DRIVE_ROOT)]
if not torch.cuda.is_available():
    command.append('--cpu')
print('Dispositivo:', 'GPU' if torch.cuda.is_available() else 'CPU', flush=True)
subprocess.run(command, cwd=REPO, check=True)

status = json.loads(status_path.read_text(encoding='utf-8'))
if status.get('state') != 'completed' or status.get('return_code') != 0:
    raise RuntimeError(f'El runner no dejo el trabajo completado: {status}')
checkpoints = sorted({str(p.relative_to(WORK)) for p in WORK.rglob('best_model.pth')})
if not checkpoints:
    raise FileNotFoundError('Entrenamiento reportado como completo pero no existe best_model.pth en Drive.')
logs = list(WORK.rglob('training_log.csv'))
if logs:
    with logs[0].open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    epochs = sorted({row.get('epoch', '') for row in rows if row.get('epoch', '').isdigit()},
                    key=int)
    print(f"Epocas registradas: {len(epochs)}; ultima epoca: {epochs[-1] if epochs else 'sin filas'}.")
print('Estado final:', status.get('state'), '| checkpoints Drive:', checkpoints)
print('MVC selective-polarity finalizado.')
